In [ ]:
pip install pandas scikit-learn xgboost seaborn matplotlib

In [ ]:
# ====================================================
# 1. IMPORT LIBRARIES
# ====================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier


# ====================================================
# 2. LOAD DATASET
# ====================================================
df = pd.read_csv("/industry.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.info())


# ====================================================
# 3. DATA PREPROCESSING
# ====================================================

# Encode categorical columns
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# Choose target column
# Example: task_label or anomaly_flag depending on dataset
# Changed target column from df.columns[-1] (img_feat_128) to 'label' for classification
target_column = 'label'

X = df.drop(columns=[target_column])
y = df[target_column]


# ====================================================
# 4. FEATURE SCALING
# ====================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ====================================================
# 5. TRAIN TEST SPLIT
# ====================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ====================================================
# 6. RANDOM FOREST
# ====================================================
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)


# ====================================================
# 7. BAGGING
# ====================================================
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=150,
    random_state=42
)

bag.fit(X_train, y_train)
bag_pred = bag.predict(X_test)


# ====================================================
# 8. BOOSTING (ADABOOST)
# ====================================================
ada = AdaBoostClassifier(
    n_estimators=150,
    learning_rate=0.1
)

ada.fit(X_train, y_train)
ada_pred = ada.predict(X_test)


# ====================================================
# 9. GRADIENT BOOSTING
# ====================================================
gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1
)

gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)


# ====================================================
# 10. XGBOOST
# ====================================================
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)


# ====================================================
# 11. VOTING ENSEMBLE
# ====================================================
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('bag', bag),
        ('gb', gb),
        ('xgb', xgb)
    ],
    voting='soft'
)

ensemble.fit(X_train, y_train)
ens_pred = ensemble.predict(X_test)


# ====================================================
# 12. EVALUATION FUNCTION
# ====================================================
def evaluate_model(name, y_true, y_pred):
    print("\n", name)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))


evaluate_model("Random Forest", y_test, rf_pred)
evaluate_model("Bagging", y_test, bag_pred)
evaluate_model("AdaBoost", y_test, ada_pred)
evaluate_model("Gradient Boosting", y_test, gb_pred)
evaluate_model("XGBoost", y_test, xgb_pred)
evaluate_model("Voting Ensemble", y_test, ens_pred)

Dataset shape: (10000, 135)
         image_id  force_sensor  proximity_sensor  temperature_sensor  \
0  img_000001.jpg      0.374540          0.373641           63.799899   
1  img_000002.jpg      0.950714          0.332912           31.070720   
2  img_000003.jpg      0.731994          0.176154           40.798382   
3  img_000004.jpg      0.598658          0.607267           59.796838   
4  img_000005.jpg      0.156019          0.476624           48.925361   

  task_type object_class  label  img_feat_1  img_feat_2  img_feat_3  ...  \
0  assemble     cylinder      1    0.754303    0.390235    0.406248  ...   
1      weld         gear      0    0.204475    0.282728    0.444483  ...   
2      pick        panel      1    0.466118    0.936893    0.564633  ...   
3  assemble      bearing      0    0.418910    0.614430    0.811354  ...   
4  assemble        panel      1    0.876187    0.130566    0.699327  ...   

   img_feat_119  img_feat_120  img_feat_121  img_feat_122  img_feat_123  \
0

# New Section